# Day 2 — Retrieval Optimization
### AI Clinical Decision Support Lite · AI Max Team

This notebook documents our **Day 2 deliverables** on the 190-chunk WHO index:

1. **Top-K tuning** — selected `K=3` via Precision@K / PageRecall@K / DocHit@K
2. **Chunk ablation** — validated 500/75 production config
3. **Evaluation benchmark** — `evaluation_set.py` (10 positive + 2 negative queries)
4. **Robustness check** — paraphrase held-out set (anti-overfitting)
5. **Explainability** — `retrieve_with_citation()` + confidence labels

Imports match production scripts: `evaluate_retrieval.py`, `evaluation_set.py`, `retrieval.py`.

## 0. Setup — Load Persisted Index

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "notebooks").exists() and not (REPO_ROOT / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent if REPO_ROOT.name == "day2" else REPO_ROOT
if not (REPO_ROOT / "config.py").exists():
    REPO_ROOT = Path(os.path.abspath("../../"))
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import config
from query import load_index, retrieve
from ingest import load_pdfs, chunk_documents

# Ensure index exists (run Day 1 notebook or `python ingest.py` first)
pages = load_pdfs(config.DATA_DIR)
expected_chunks = len(chunk_documents(pages))
vectordb = load_index()
count = vectordb._collection.count()

print(f"✅ Vector index loaded: {count} chunks (pipeline expects {expected_chunks})")
print(f"   TOP_K={config.TOP_K} | CONFIDENCE_THRESHOLD={config.CONFIDENCE_THRESHOLD}")
print(f"   Corpus: {len(pages)} pages · 2 WHO premier guidelines")

✅ Vector index loaded: 190 chunks (pipeline expects 190)
   TOP_K=3 | CONFIDENCE_THRESHOLD=0.7
   Corpus: 104 pages · 2 WHO premier guidelines


## 1. Top-K Tuning Demo

We compare $k \in \{1, 3, 8\}$ on our anchor clinical query, then run the full sweep via `evaluate_retrieval.py`.

In [2]:
question = "What is the target blood pressure for a patient with cardiovascular disease?"

for k in [1, 3, 8]:
    results = retrieve(vectordb, question, k=k)
    print(f"{'='*20} k={k} {'='*20}")
    for rank, (doc, score) in enumerate(results, 1):
        print(f"  [{rank}] score={score:.3f} | p.{doc.metadata.get('page_number')} | "
              f"{doc.metadata.get('document_name','')[:45]}")
    print()

==================== k=1 ====================
  [1] score=0.793 | p.28 | Guideline for the pharmacological treatment o

==================== k=3 ====================
  [1] score=0.793 | p.28 | Guideline for the pharmacological treatment o
  [2] score=0.746 | p.14 | WHO-NMH-NVI-18.2-eng.pdf
  [3] score=0.740 | p.11 | Guideline for the pharmacological treatment o

==================== k=8 ====================
  [1] score=0.793 | p.28 | Guideline for the pharmacological treatment o
  [2] score=0.746 | p.14 | WHO-NMH-NVI-18.2-eng.pdf
  [3] score=0.740 | p.11 | Guideline for the pharmacological treatment o
  [4] score=0.690 | p.13 | Guideline for the pharmacological treatment o
  [5] score=0.670 | p.9 | Guideline for the pharmacological treatment o
  [6] score=0.657 | p.12 | WHO-NMH-NVI-18.2-eng.pdf
  [7] score=0.656 | p.19 | Guideline for the pharmacological treatment o
  [8] score=0.647 | p.24 | Guideline for the pharmacological treatment o



In [3]:
from evaluate_retrieval import evaluate_k_values
from evaluation_set import POSITIVE_EVAL_SET, NEGATIVE_EVAL_SET

metrics = evaluate_k_values(vectordb)
print(f"{'K':>4} | {'Precision@K':>12} | {'PageRecall@K':>13} | {'DocHit@K':>10}")
print("-" * 48)
for m in metrics:
    flag = " ← SELECTED" if m["k"] == config.TOP_K else ""
    print(f"{m['k']:>4} | {m['precision']:>12.2%} | {m['page_recall']:>13.2%} | {m['doc_hit']:>10.2%}{flag}")

opt = next(m for m in metrics if m["k"] == config.TOP_K)
print(f"\nDecision: K={config.TOP_K} — PageRecall={opt['page_recall']:.0%}, Precision={opt['precision']:.1%}")

   K |  Precision@K |  PageRecall@K |   DocHit@K
------------------------------------------------
   1 |       80.00% |        80.00% |    100.00%
   3 |       63.33% |       100.00% |    100.00% ← SELECTED
   4 |       55.00% |       100.00% |    100.00%
   5 |       46.00% |       100.00% |    100.00%
  10 |       28.00% |       100.00% |    100.00%

Decision: K=3 — PageRecall=100%, Precision=63.3%


### Checkpoint 1
- **K=1** maximizes first-result precision but may miss supporting evidence.
- **K=3** (our choice) balances **100% PageRecall@3** with acceptable precision for LLM context.
- **K=8** introduces noise from background pages — visible in live scores above.

## 2. Chunk Size Ablation (Module 2)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from ingest import get_embedding_function
from evaluation_set import POSITIVE_EVAL_SET

configs = [
    {"name": "300/40",  "size": 300, "overlap": 40},
    {"name": "500/75*", "size": 500, "overlap": 75},
    {"name": "700/100", "size": 700, "overlap": 100},
]
embed_fn = get_embedding_function()

print(f"{'Config':<10} | {'Chunks':>6} | {'Precision@5':>12}")
print("-" * 36)
for cfg in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["size"] * config.CHARS_PER_TOKEN,
        chunk_overlap=cfg["overlap"] * config.CHARS_PER_TOKEN,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    test_chunks = splitter.split_documents(pages)
    tmp = Chroma.from_documents(
        test_chunks, embed_fn,
        collection_name=f"nb_ablation_{cfg['name'].replace('/','_')}",
        ids=[f"tmp_{i}" for i in range(len(test_chunks))],
    )
    scores = []
    for item in POSITIVE_EVAL_SET:
        res = tmp.similarity_search_with_relevance_scores(item["question"], k=5)
        hits = sum(1 for d, _ in res
                   if d.metadata.get("document_name") == item["expected_document"]
                   and d.metadata.get("page_number") in item["expected_pages"])
        scores.append(hits / 5)
    p5 = sum(scores) / len(scores)
    tmp.delete_collection()
    mark = " ← production" if cfg["name"].startswith("500") else ""
    print(f"{cfg['name']:<10} | {len(test_chunks):>6} | {p5:>11.1%}{mark}")

Config     | Chunks |  Precision@5
------------------------------------
300/40     |    294 |       52.0%
500/75*    |    190 |       60.0% ← production
700/100    |    142 |       48.0%


## 3. Reference Evaluation Set

In [5]:
from evaluation_set import EVAL_SET, export_eval_csv

export_eval_csv()
print(f"EVAL_SET: {len(EVAL_SET)} queries ({len(POSITIVE_EVAL_SET)} positive, {len(NEGATIVE_EVAL_SET)} negative)\n")
for i, item in enumerate(POSITIVE_EVAL_SET, 1):
    pages_str = ", ".join(str(p) for p in item["expected_pages"][:4])
    suffix = "..." if len(item["expected_pages"]) > 4 else ""
    print(f"[{i:02d}] [{item['difficulty']:>6}] {item['question'][:58]}")
    print(f"      → {item['expected_document'][:50]} | pages [{pages_str}{suffix}]\n")

EVAL_SET: 12 queries (10 positive, 2 negative)

[01] [  easy] What is the target blood pressure for a patient with cardi
      → Guideline for the pharmacological treatment of hyp | pages [28]

[02] [medium] Which antihypertensive drug classes are recommended as fir
      → Guideline for the pharmacological treatment of hyp | pages [10, 32, 33]

[03] [medium] What are the blood pressure thresholds for initiating anti
      → Guideline for the pharmacological treatment of hyp | pages [9, 19]

[04] [  hard] What lifestyle modifications are recommended alongside pha
      → Guideline for the pharmacological treatment of hyp | pages [14, 15]

[05] [medium] When should combination therapy be initiated for patients 
      → Guideline for the pharmacological treatment of hyp | pages [26, 38, 39]

[06] [medium] What is the recommended follow-up interval for blood press
      → Guideline for the pharmacological treatment of hyp | pages [30, 37, 38]

[07] [  easy] What is the HEARTS treatment pr

## 4. Per-Question Retrieval at K=3

In [6]:
from evaluate_retrieval import _page_hits, _doc_hit

print(f"{'Status':<6} | {'P@3':>5} | {'Top':>6} | Question")
print("-" * 75)
for item in POSITIVE_EVAL_SET:
    res = retrieve(vectordb, item["question"])
    hits = _page_hits(res, item)
    top = res[0][1] if res else 0
    mark = "✅" if hits > 0 else "❌"
    print(f"{mark}     | {hits/config.TOP_K:>4.0%} | {top:>6.3f} | {item['question'][:48]}")

Status |   P@3 |    Top | Question
---------------------------------------------------------------------------
✅     |  33% |  0.793 | What is the target blood pressure for a patient 
✅     |  67% |  0.725 | Which antihypertensive drug classes are recommen
✅     |  33% |  0.815 | What are the blood pressure thresholds for initi
✅     |  33% |  0.751 | What lifestyle modifications are recommended alo
✅     |  33% |  0.759 | When should combination therapy be initiated for
✅     | 100% |  0.753 | What is the recommended follow-up interval for b
✅     | 100% |  0.756 | What is the HEARTS treatment protocol for hypert
✅     |  33% |  0.726 | How does the HEARTS module recommend organizing 
✅     | 100% |  0.651 | What simplified medication titration algorithm i
✅     | 100% |  0.769 | How should cardiovascular risk assessment be int


## 5. Anti-Overfitting — Paraphrase Sample

In [7]:
from evaluation_set_robustness import PARAPHRASE_EVAL_SET

print("Held-out paraphrase queries (different wording, same clinical intent):\n")
par_hits = 0
for item in PARAPHRASE_EVAL_SET[:5]:
    res = retrieve(vectordb, item["question"])
    hits = _page_hits(res, item)
    par_hits += int(hits > 0)
    mark = "✅" if hits > 0 else "❌"
    print(f"{mark} {item['question'][:65]}")

print(f"\nSample paraphrase PageRecall (first 5): {par_hits}/5 = {par_hits/5:.0%}")
print("Full suite: python evaluate_robustness.py")

Held-out paraphrase queries (different wording, same clinical intent):

✅ For adults with established heart disease, what systolic blood pr
✅ Which classes of drugs should clinicians pick first when starting
❌ At what blood pressure readings should antihypertensive medicatio
❌ Apart from medicines, what behavioral changes does WHO recommend 
✅ Under what circumstances should two antihypertensive agents be st

Sample paraphrase PageRecall (first 5): 3/5 = 60%
Full suite: python evaluate_robustness.py


## 6. Out-of-Scope Negative Controls

In [8]:
from evaluate_retrieval import evaluate_negative_controls

neg = evaluate_negative_controls(vectordb)
for r in neg:
    mark = "✅" if r["passed"] else "❌"
    print(f"{mark} top={r['top_score']:.3f} (max {r['max_allowed']:.2f}) | {r['question'][:55]}")

✅ top=0.615 (max 0.65) | What screening interval does this guideline recommend f
✅ top=0.517 (max 0.65) | What is the recommended antibiotic regimen for communit


## 7. Clinical Explainability (Module 4)

In [9]:
from retrieval import retrieve_with_citation, print_retrieval_view

demo_q = "What is the recommended target blood pressure for patients with cardiovascular disease?"
results = retrieve_with_citation(vectordb, demo_q)
print_retrieval_view(demo_q, results)


 CLINICAL QUERY: What is the recommended target blood pressure for patients with cardiovascular disease?

[1] ⭐ [CONFIDENT] | Score: 0.812 | Source: Guideline for the pharmacological treatment of hypertension in adults.pdf (p. 28)
    Chunk ID: Guideline for the pharmacological treatment of hypertension in adults.pdf_p28_c1
--------------------------------------------------------------------------------
    Content: "3.6 Target blood pressure 6. RECOMMENDATION ON TARGET BLOOD PRESSURES WHO recommends a target blood pressure treatment goal of <140/90 mmHg in all patients  with hypertension without comorbidities. Strong recommendation, moderate-certainty evidence WHO recommends a target syst..."

[2] ⭐ [CONFIDENT] | Score: 0.775 | Source: Guideline for the pharmacological treatment of hypertension in adults.pdf (p. 11)
    Chunk ID: Guideline for the pharmacological treatment of hypertension in adults.pdf_p11_c1
---------------------------------------------------------------------------

## 8. Day 2 Definition of Done

- [x] **Top-K selected** with measured Precision@K / PageRecall@K (`K=3`)
- [x] **Chunk config validated** via ablation (`500/75`, ~15% overlap)
- [x] **Evaluation set** in `evaluation_set.py` + synced CSV
- [x] **Robustness** paraphrase sample + `evaluate_robustness.py`
- [x] **Explainability** via `retrieve_with_citation()` + confidence ≥ 0.70
- [x] **Automated DoD:** `python verify_day2_dod.py` (21 checks)

**Verified scripts:** `evaluate_retrieval.py` · `evaluate_embeddings.py` · `evaluate_retrieval_architectures.py`

**Ready for Day 3:** grounded LLM generation constrained to retrieved chunks.

---

### 📦 Day 2 Deliverables (تسليمات اليوم الثاني)

- **The Notebook:** Day2_Retrieval_Optimization.ipynb
- **The Report:** DAY2_REPORT.md
- **Evaluation Data:** evaluation_set.py
- **Validation Scripts:** evaluate_retrieval.py, chunk_ablation.py, evaluate_embeddings.py, erify_day2_dod.py
